# ROME + Qwen3.5 fixed notebook v6

Run the cells in order from a fresh runtime. This version fixes Kaggle/Colab compatibility problems for text-only ROME + Qwen3.5:

1. `KeyError: qwen3_5` by installing source Transformers into `/kaggle/working/pydeps`.
2. `tokenizers==0.23.1` incompatibility by pinning `tokenizers==0.22.1`.
3. Broken `torchvision` import by forcing Transformers text-only behavior.
4. `GET was unable to find an engine` by purging copied `torch`/CUDA packages from `/kaggle/working/pydeps` and using Kaggle's built-in `torch`.
5. Qwen3.5 ROME compatibility by using Qwen3.5 module paths and a stable no-generated-context smoke-test config.
6. EasyEdit/PyTorch hook compatibility: fixes `register_forward_hook(..., with_kwargs=True)` argument order so ROME hooks return tensors, not kwargs dicts.


In [1]:
# Cell 1 - Clone EasyEdit fresh
%cd /kaggle/working
!rm -rf EasyEdit
!git clone https://github.com/zjunlp/EasyEdit.git
%cd /kaggle/working/EasyEdit


/kaggle/working
Cloning into 'EasyEdit'...
remote: Enumerating objects: 10275, done.
remote: Counting objects: 100% (2021/2021), done.
remote: Compressing objects: 100% (562/562), done.
remote: Total 10275 (delta 1627), reused 1459 (delta 1459), pack-reused 8254 (from 2)
Receiving objects: 100% (10275/10275), 96.41 MiB | 44.15 MiB/s, done.
Resolving deltas: 100% (6617/6617), done.
/kaggle/working/EasyEdit


In [2]:
# Cell 2 - Install text-only dependencies, but keep Kaggle's built-in torch
# IMPORTANT: after this cell, run Cell 3 before importing transformers/easyeditor.
# IMPORTANT: do NOT install torch/torchvision/torchaudio into /kaggle/working/pydeps.
# This cell intentionally pins the source-Transformers runtime dependencies.

!rm -rf /kaggle/working/pydeps
!mkdir -p /kaggle/working/pydeps

# ROME + Qwen text-only editing does not need vision/audio packages.
# Remove them because mismatched torchvision/torchaudio can break text-model imports.
!python -m pip uninstall -y torchvision torchaudio || true

# EasyEdit/ROME text-only dependencies. These should not install torch.
!python -m pip install --no-cache-dir --upgrade --target /kaggle/working/pydeps "numpy==1.26.4" "scipy==1.13.1" "scikit-learn==1.5.2" "pandas==2.2.3" "datasets>=2.15.0" "einops>=0.8.0" "hydra-core>=1.3.2" "omegaconf>=2.3.0" "nltk==3.8.1" "rouge==1.0.1"

# Runtime dependencies required by the current Transformers source tree.
# Do NOT leave tokenizers unconstrained: tokenizers==0.23.1 fails the Transformers import check.
!python -m pip install --no-cache-dir --upgrade --target /kaggle/working/pydeps "huggingface-hub>=1.5.0,<2.0" "tokenizers==0.22.1" "regex>=2025.10.22" "safetensors>=0.4.3" "packaging>=20.0" "pyyaml>=5.1" "tqdm>=4.27" "typer" "filelock" "requests"

# Qwen3.5 currently needs Transformers source/main in many notebook images.
# --no-deps prevents pip from installing another torch or a newer incompatible tokenizers.
!python -m pip install --no-cache-dir --upgrade --target /kaggle/working/pydeps --no-deps "git+https://github.com/huggingface/transformers.git@main"

# Hard purge: never let Python import torch/CUDA libraries from pydeps.
# The earlier traceback showed /kaggle/working/pydeps/torch, which is the broken path.
!rm -rf /kaggle/working/pydeps/torch /kaggle/working/pydeps/torch-* /kaggle/working/pydeps/torchvision* /kaggle/working/pydeps/torchaudio* /kaggle/working/pydeps/triton* /kaggle/working/pydeps/nvidia* /kaggle/working/pydeps/functorch* /kaggle/working/pydeps/caffe2* || true

print('Done. Cell 3 will verify torch/tokenizers/transformers paths and versions.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 74.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 69.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 235.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 258.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 174.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 282.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 310.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 221.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 229.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 192.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

In [3]:
# Cell 3 - Set paths and environment before all imports
import os
import sys
import gc
import re
import json
import time
import shutil
from pathlib import Path
from packaging.version import parse as V

PYDEPS = "/kaggle/working/pydeps"
REPO = "/kaggle/working/EasyEdit"

# Extra safety: remove any copied torch/CUDA packages before adding PYDEPS to sys.path.
# We want Kaggle's built-in torch, not a pip-copied torch from pydeps.
for pattern in [
    "torch", "torch-*", "torchvision*", "torchaudio*", "triton*",
    "nvidia*", "functorch*", "caffe2*",
]:
    for p in Path(PYDEPS).glob(pattern):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
        elif p.exists():
            p.unlink()

# Put pydeps first so the source Transformers installed in Cell 2 wins.
# Since torch was purged from pydeps, import torch will fall through to Kaggle's working CUDA build.
for p in [REPO, PYDEPS]:
    if p in sys.path:
        sys.path.remove(p)
for p in [PYDEPS, REPO]:
    sys.path.insert(0, p)

os.environ["PYTHONPATH"] = f"{REPO}:{PYDEPS}:" + os.environ.get("PYTHONPATH", "")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Force Transformers to behave as text-only even if a broken global torchvision exists.
os.environ["USE_TORCHVISION"] = "0"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

# Clear stale modules if this cell is rerun after a failed import.
for name in list(sys.modules.keys()):
    if name.startswith((
        "transformers", "tokenizers", "huggingface_hub", "easyeditor",
        "torch", "torchvision", "torchaudio", "triton"
    )):
        del sys.modules[name]

print("sys.path[:5] =", sys.path[:5])
print("PYDEPS:", PYDEPS)

# Verify tokenizers BEFORE importing transformers. This catches the exact error from Cell 4.
import tokenizers
print("Tokenizers version:", tokenizers.__version__)
print("Tokenizers file:", tokenizers.__file__)
if not (V("0.22.0") <= V(tokenizers.__version__) <= V("0.23.0")):
    raise RuntimeError(
        f"Wrong tokenizers version: {tokenizers.__version__} from {tokenizers.__file__}\n"
        "Run Cell 2 again. It must install tokenizers==0.22.1 into /kaggle/working/pydeps."
    )
if not str(tokenizers.__file__).startswith(PYDEPS):
    raise RuntimeError(
        f"Wrong tokenizers path: {tokenizers.__file__}\n"
        "The pydeps tokenizers must come first in sys.path. Restart runtime and rerun Cell 2/3."
    )

# Verify torch path immediately. It must NOT be /kaggle/working/pydeps/torch.
import torch
print("Torch version:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("Torch file:", torch.__file__)
print("CUDA available:", torch.cuda.is_available())
if str(torch.__file__).startswith(PYDEPS):
    raise RuntimeError(
        f"Wrong torch path: {torch.__file__}\n"
        "Restart runtime, rerun Cell 2, then Cell 3. Torch must come from Kaggle's base environment."
    )

# Qwen3.5 has Conv1d layers in linear attention. If cuDNN engine selection fails on Kaggle,
# this forces PyTorch to use a safer non-cuDNN path for those Conv1d calls.
torch.backends.cudnn.enabled = False
print("cuDNN enabled:", torch.backends.cudnn.enabled)


def disable_torchvision_for_transformers():
    # Prevent Transformers text models from importing broken global torchvision.
    import transformers.utils.import_utils as hf_import_utils
    hf_import_utils._torchvision_available = False
    hf_import_utils._torchvision_version = "N/A"
    if "transformers.image_utils" in sys.modules:
        try:
            sys.modules["transformers.image_utils"].is_torchvision_available = lambda: False
        except Exception:
            pass
    return True

print("REPO:", REPO)


sys.path[:5] = ['/kaggle/working/EasyEdit', '/kaggle/working/pydeps', '/kaggle/working', '/kaggle/lib/kagglegym', '/kaggle/lib']
PYDEPS: /kaggle/working/pydeps
Tokenizers version: 0.22.1
Tokenizers file: /kaggle/working/pydeps/tokenizers/__init__.py
Torch version: 2.10.0+cu128
Torch CUDA: 12.8
Torch file: /usr/local/lib/python3.12/dist-packages/torch/__init__.py
CUDA available: True
cuDNN enabled: False
REPO: /kaggle/working/EasyEdit


In [4]:
# Cell 4 - Verify Transformers now recognizes qwen3_5
import os
import sys
from packaging.version import parse as V

# Make sure the failed-import version problem cannot silently continue.
for name in list(sys.modules.keys()):
    if name.startswith(("transformers", "tokenizers")):
        del sys.modules[name]

import tokenizers
print("Tokenizers version before Transformers import:", tokenizers.__version__)
print("Tokenizers file before Transformers import:", tokenizers.__file__)
if not (V("0.22.0") <= V(tokenizers.__version__) <= V("0.23.0")):
    raise RuntimeError("tokenizers must be >=0.22.0 and <=0.23.0. Re-run Cell 2, then Cell 3.")

import transformers
disable_torchvision_for_transformers()
from transformers import AutoConfig

MODEL_ID = "Qwen/Qwen3.5-4B-Base"  # Better for ROME than the post-trained chat model.
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

print("Transformers version:", transformers.__version__)
print("Transformers file:", transformers.__file__)
if not str(transformers.__file__).startswith(PYDEPS):
    raise RuntimeError(f"Wrong Transformers path: {transformers.__file__}. It should come from /kaggle/working/pydeps.")

cfg = AutoConfig.from_pretrained(MODEL_ID, token=hf_token, trust_remote_code=True)
print("top-level model_type:", cfg.model_type)
print("text model_type:", getattr(getattr(cfg, "text_config", None), "model_type", None))
print("architectures:", getattr(cfg, "architectures", None))


Tokenizers version before Transformers import: 0.22.1
Tokenizers file before Transformers import: /kaggle/working/pydeps/tokenizers/__init__.py
Transformers version: 5.8.0.dev0
Transformers file: /kaggle/working/pydeps/transformers/__init__.py
top-level model_type: qwen3_5
text model_type: qwen3_5_text
architectures: ['Qwen3_5ForConditionalGeneration']


In [5]:
# Cell 5 - Hugging Face login and ZsRE data
from huggingface_hub import login, hf_hub_download

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN").strip()
except Exception:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
    login(token=hf_token)
    print("HF login OK")
else:
    print("No HF token found. Public downloads may still work.")

DATA_DIR = Path("/kaggle/working/EasyEdit/data/zsre_real")
DATA_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = DATA_DIR / "zsre_mend_train_10000.json"
EVAL_PATH = DATA_DIR / "zsre_mend_eval_portability_gpt4.json"

train_src = hf_hub_download(
    repo_id="wangzn2001/data",
    repo_type="dataset",
    filename="data/zsre/zsre_mend_train_10000.json",
    token=hf_token,
)
eval_src = hf_hub_download(
    repo_id="wangzn2001/data",
    repo_type="dataset",
    filename="data/portability/One Hop/zsre_mend_eval_portability_gpt4.json",
    token=hf_token,
)
shutil.copy(train_src, TRAIN_PATH)
shutil.copy(eval_src, EVAL_PATH)
print("Downloaded:", TRAIN_PATH, EVAL_PATH)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK
Downloaded: /kaggle/working/EasyEdit/data/zsre_real/zsre_mend_train_10000.json /kaggle/working/EasyEdit/data/zsre_real/zsre_mend_eval_portability_gpt4.json


In [6]:
# Cell 6 - Clean EasyEdit text-only ROME patches
# Run BEFORE any easyeditor import.
REPO_PATH = Path("/kaggle/working/EasyEdit")

(REPO_PATH / "easyeditor/__init__.py").write_text('''
from .models.rome.rome_hparams import ROMEHyperParams
from .editors.editor import BaseEditor
'''.strip() + "\n")

(REPO_PATH / "easyeditor/editors/__init__.py").write_text('''
from .editor import BaseEditor
'''.strip() + "\n")

(REPO_PATH / "easyeditor/evaluate/__init__.py").write_text('''
from .evaluate import *
from .evaluate_utils import *

def compute_sent_metric(*args, **kwargs):
    return {}
'''.strip() + "\n")

(REPO_PATH / "easyeditor/trainer/__init__.py").write_text("# Text-only ROME/Qwen patch.\n")
(REPO_PATH / "easyeditor/trainer/algs/__init__.py").write_text("# Text-only ROME/Qwen patch.\n")
(REPO_PATH / "easyeditor/models/__init__.py").write_text('''
from .rome.rome_hparams import ROMEHyperParams
'''.strip() + "\n")

(REPO_PATH / "easyeditor/util/alg_dict.py").write_text('''
from ..models.rome import ROMEHyperParams, apply_rome_to_model

ALG_DICT = {"ROME": apply_rome_to_model}
ALG_HPARAMS = {"ROME": ROMEHyperParams}
DS_DICT = {}
ALG_MULTIMODAL_DICT = {}
PER_ALG_DICT = {}
MULTIMODAL_DS_DICT = {}
PER_DS_DICT = {}
Safety_DS_DICT = {}
'''.strip() + "\n")

trainer_models = REPO_PATH / "easyeditor/trainer/models.py"
if trainer_models.exists():
    text = trainer_models.read_text()
    text = text.replace(
        "from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration, Qwen2VLForConditionalGeneration",
        "# Disabled for text-only ROME/Qwen run:\nAutoProcessor = None\nLlavaOnevisionForConditionalGeneration = None\nQwen2VLForConditionalGeneration = None",
    )
    text = text.replace(
        "from transformers import LlavaOnevisionForConditionalGeneration, Qwen2VLForConditionalGeneration",
        "# Disabled for text-only ROME/Qwen run:\nLlavaOnevisionForConditionalGeneration = None\nQwen2VLForConditionalGeneration = None",
    )
    trainer_models.write_text(text)

editor_path = REPO_PATH / "easyeditor/editors/editor.py"
text = editor_path.read_text()
text = re.sub(
    r"^from\s+\.\.models\.melo\.melo\s+import\s+LORA\s*$",
    "# Disabled for text-only ROME/Qwen run; MELO pulls PEFT/vision/torchvision paths.\nLORA = None",
    text,
    flags=re.MULTILINE,
)
text = re.sub(
    r"^from\s+\.models\.melo\.melo\s+import\s+LORA\s*$",
    "# Disabled for text-only ROME/Qwen run; MELO pulls PEFT/vision/torchvision paths.\nLORA = None",
    text,
    flags=re.MULTILINE,
)
# Remove old/nonstandard fp32=False and add safe kwargs for Qwen3.5 text loading.
text = text.replace(
    "self.model = AutoModelForCausalLM.from_pretrained(self.model_name,fp32=False,trust_remote_code=True, **model_kwargs)",
    "self.model = AutoModelForCausalLM.from_pretrained(\n"
    "                    self.model_name,\n"
    "                    trust_remote_code=True,\n"
    "                    dtype=torch.float16,\n"
    "                    attn_implementation='eager',\n"
    "                    **model_kwargs\n"
    "                )",
)
# If the cloned editor uses a slightly different call, patch it too.
text = text.replace(
    "self.model = AutoModelForCausalLM.from_pretrained(self.model_name, trust_remote_code=True, **model_kwargs)",
    "self.model = AutoModelForCausalLM.from_pretrained(\n"
    "                    self.model_name,\n"
    "                    trust_remote_code=True,\n"
    "                    dtype=torch.float16,\n"
    "                    attn_implementation='eager',\n"
    "                    **model_kwargs\n"
    "                )",
)

# EasyEdit later does: isinstance(edited_model, LORA).
# Because we disabled MELO/PEFT by setting LORA = None, that check must guard LORA first.
# Otherwise ROME completes successfully but crashes while returning metrics.
text = text.replace(
    "if isinstance(edited_model, LORA):",
    "if LORA is not None and isinstance(edited_model, LORA):",
)

editor_path.write_text(text)

for name in list(sys.modules.keys()):
    if name.startswith("easyeditor"):
        del sys.modules[name]

print("Text-only ROME patches applied")
!grep -n "melo\|LORA\|AutoModelForCausalLM\|attn_implementation\|dtype\|isinstance" /kaggle/working/EasyEdit/easyeditor/editors/editor.py | head -40

# Patch EasyEdit's Trace hook for modern PyTorch.
# PyTorch register_forward_hook(..., with_kwargs=True) calls:
#     hook(module, args, kwargs, output)
# Some EasyEdit versions use the older/wrong order:
#     hook(module, args, output, kwargs=None)
# which returns kwargs as the module output and breaks Qwen3.5 MLP/down_proj tracing.
nethook_path = REPO_PATH / "easyeditor/util/nethook.py"
text = nethook_path.read_text()
old_sig = "def retain_hook(m, inputs, output, kwargs=None):"
new_sig = "def retain_hook(m, inputs, kwargs, output):"
if old_sig in text:
    text = text.replace(old_sig, new_sig)
text = text.replace(
    "return retain_hook(m, inputs, output, kwargs=None)",
    "return retain_hook(m, inputs, None, output)",
)
# Preserve recursive_copy options for containers. This is not the main bug, but it makes retain_input/output safer.
text = text.replace(
    "return type(x)({k: recursive_copy(v) for k, v in x.items()})",
    "return type(x)({k: recursive_copy(v, clone=clone, detach=detach, retain_grad=retain_grad) for k, v in x.items()})",
)
text = text.replace(
    "return type(x)([recursive_copy(v) for v in x])",
    "return type(x)([recursive_copy(v, clone=clone, detach=detach, retain_grad=retain_grad) for v in x])",
)
nethook_path.write_text(text)

# Sanity check the exact hook behavior that broke Qwen3.5.
import torch
from easyeditor.util import nethook

class _KwargOnlyModule(torch.nn.Module):
    def forward(self, hidden_states=None):
        return hidden_states + 1

_test_mod = _KwargOnlyModule()
with nethook.Trace(_test_mod, retain_input=True, retain_output=True) as tr:
    _out = _test_mod(hidden_states=torch.ones(1, 2))

assert torch.is_tensor(_out), f"nethook returned {type(_out)} instead of Tensor"
assert torch.is_tensor(tr.output), f"nethook retained output as {type(tr.output)} instead of Tensor"
assert torch.is_tensor(tr.input), f"nethook retained input as {type(tr.input)} instead of Tensor"
print("nethook with_kwargs patch OK; hook output stayed Tensor.")


editor_text_after = editor_path.read_text()
assert "if LORA is not None and isinstance(edited_model, LORA):" in editor_text_after, "LORA isinstance guard was not patched"
print("LORA isinstance guard patch OK.")


Text-only ROME patches applied
9:LORA = None
10:from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig
66:            torch_dtype = torch.float16 if hasattr(hparams, 'fp16') and hparams.fp16 else torch.float32
74:                    bnb_4bit_compute_dtype=torch.bfloat16
78:                    "torch_dtype": torch_dtype,
83:                    "torch_dtype": torch_dtype,
97:                self.model = AutoModelForCausalLM.from_pretrained(self.model_name, **model_kwargs)
101:                self.model = AutoModelForCausalLM.from_pretrained(self.model_name, **model_kwargs)
105:                self.model = AutoModelForCausalLM.from_pretrained(self.model_name, **model_kwargs, trust_remote_code=True)
120:                self.model = AutoModelForCausalLM.from_pretrained(self.model_name,trust_remote_code=True, torch_dtype=torch_dtype if hparams.alg_name not in ['MEND'] else torch.bfloat16, device_map=device_map)
123:                self.model = AutoModelFo

/kaggle/working/EasyEdit/easyeditor/models/melo/peft_egg/src/peft/tuners/lora.py:233: SyntaxWarning: invalid escape sequence '\.'
  layer_index = re.match(f".*.{pattern}\.(\d+)\.*", key)
/kaggle/working/EasyEdit/easyeditor/models/melo/peft_egg/src/peft/tuners/melo.py:183: SyntaxWarning: invalid escape sequence '\.'
  layer_index = re.match(f".*.{pattern}\.(\d+)\.*", key)


nethook with_kwargs patch OK; hook output stayed Tensor.
LORA isinstance guard patch OK.


In [7]:
# Cell 7 - Create Qwen3.5-4B-Base ROME YAML
ROME_QWEN_YAML = Path("/kaggle/working/EasyEdit/hparams/ROME/qwen35_4b_base_zsre_kaggle.yaml")
ROME_QWEN_YAML.parent.mkdir(parents=True, exist_ok=True)
ROME_QWEN_YAML.write_text('''
alg_name: ROME
model_name: Qwen/Qwen3.5-4B-Base
stats_dir: /kaggle/working/EasyEdit/data/stats
device: 0
layers: [17]
fact_token: subject_last
v_num_grad_steps: 25
v_lr: 3e-1
v_loss_layer: 31
v_weight_decay: 1e-3
clamp_norm_factor: 4
kl_factor: 0.0625
mom2_adjustment: false
# Stable Kaggle smoke-test mode: avoid generate_fast context generation, which can trigger
# Qwen3.5 linear-attention Conv1d/cuDNN issues on some notebook images.
# After the smoke test works, you may try [[5, 10], [10, 10]] for stronger ROME contexts.
context_template_length_params: []
rewrite_module_tmp: model.layers.{}.mlp.down_proj
layer_module_tmp: model.layers.{}
mlp_module_tmp: model.layers.{}.mlp
attn_module_tmp: model.layers.{}.self_attn
ln_f_module: model.norm
lm_head_module: lm_head
mom2_dataset: wikipedia
mom2_n_samples: 1000
mom2_dtype: float32
model_parallel: false
fp16: true
max_length: 40
'''.strip() + "\n")
print(ROME_QWEN_YAML.read_text())


alg_name: ROME
model_name: Qwen/Qwen3.5-4B-Base
stats_dir: /kaggle/working/EasyEdit/data/stats
device: 0
layers: [17]
fact_token: subject_last
v_num_grad_steps: 25
v_lr: 3e-1
v_loss_layer: 31
v_weight_decay: 1e-3
clamp_norm_factor: 4
kl_factor: 0.0625
mom2_adjustment: false
# Stable Kaggle smoke-test mode: avoid generate_fast context generation, which can trigger
# Qwen3.5 linear-attention Conv1d/cuDNN issues on some notebook images.
# After the smoke test works, you may try [[5, 10], [10, 10]] for stronger ROME contexts.
context_template_length_params: []
rewrite_module_tmp: model.layers.{}.mlp.down_proj
layer_module_tmp: model.layers.{}
mlp_module_tmp: model.layers.{}.mlp
attn_module_tmp: model.layers.{}.self_attn
ln_f_module: model.norm
lm_head_module: lm_head
mom2_dataset: wikipedia
mom2_n_samples: 1000
mom2_dtype: float32
model_parallel: false
fp16: true
max_length: 40



In [8]:
# Cell 8 - Verify layer names, then load ROME editor
import torch
print("Torch file in Cell 8:", torch.__file__)
if str(torch.__file__).startswith("/kaggle/working/pydeps"):
    raise RuntimeError("Wrong torch was imported from pydeps. Restart and rerun Cell 2/3.")
torch.backends.cudnn.enabled = False
disable_torchvision_for_transformers()
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3.5-4B-Base"
print("Torchvision disabled for Transformers:", True)
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token, trust_remote_code=True)

from easyeditor import ROMEHyperParams, BaseEditor
hparams = ROMEHyperParams.from_hparams(str(ROME_QWEN_YAML))
editor = BaseEditor.from_hparams(hparams)
print("Qwen3.5-4B-Base ROME editor loaded")

# Verify Qwen3.5 module paths on the actual model that ROME will edit.
model = editor.model
print("num layers:", len(model.model.layers))
print("layer 17 type:", getattr(model.model.layers[17], "layer_type", "unknown"))
for n, _ in model.named_parameters():
    if ".mlp.down_proj.weight" in n:
        print("example rewrite param:", n)
        break

!nvidia-smi


05/05/2026 03:16:05 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B-Base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/05/2026 03:16:05 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B-Base/1001bb4d826a52d1f399e183466143f4da7b741b/config.json "HTTP/1.1 200 OK"


Torch file in Cell 8: /usr/local/lib/python3.12/dist-packages/torch/__init__.py
Torchvision disabled for Transformers: True


05/05/2026 03:16:05 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B-Base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
05/05/2026 03:16:05 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B-Base/1001bb4d826a52d1f399e183466143f4da7b741b/tokenizer_config.json "HTTP/1.1 200 OK"
05/05/2026 03:16:06 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B-Base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
05/05/2026 03:16:06 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B-Base/1001bb4d826a52d1f399e183466143f4da7b741b/tokenizer_config.json "HTTP/1.1 200 OK"
05/05/2026 03:16:06 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3.5-4B-Base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
05/05/2026 03:16:06 - INFO - httpx -   H

We are creating the logger files


05/05/2026 03:16:07 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B-Base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
05/05/2026 03:16:07 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B-Base/resolve/main/model.safetensors.index.json "HTTP/1.1 307 Temporary Redirect"
05/05/2026 03:16:07 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B-Base/1001bb4d826a52d1f399e183466143f4da7b741b/model.safetensors.index.json "HTTP/1.1 200 OK"
05/05/2026 03:16:07 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3.5-4B-Base/revision/main "HTTP/1.1 200 OK"


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

05/05/2026 03:16:32 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B-Base/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
05/05/2026 03:16:32 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B-Base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/05/2026 03:16:32 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B-Base/1001bb4d826a52d1f399e183466143f4da7b741b/config.json "HTTP/1.1 200 OK"
05/05/2026 03:16:32 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B-Base/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
05/05/2026 03:16:33 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B-Base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/05/2026 03:16:33 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B-Base/1001bb4d826a52d1f399e1

Qwen3.5-4B-Base ROME editor loaded
num layers: 32
layer 17 type: linear_attention
example rewrite param: model.layers.0.mlp.down_proj.weight
Tue May  5 03:16:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P0             28W /   70W |    8135MiB /  15360MiB |     16%      Default |

In [9]:
# Cell 9 - One-edit smoke test
import json
import torch
import gc
print("Torch file in smoke test:", torch.__file__)
if str(torch.__file__).startswith("/kaggle/working/pydeps"):
    raise RuntimeError("Wrong torch was imported from pydeps. Restart and rerun Cell 2/3.")
torch.backends.cudnn.enabled = False

eval_path = "/kaggle/working/EasyEdit/data/zsre_real/zsre_mend_eval_portability_gpt4.json"
eval_items = json.load(open(eval_path))
x = eval_items[0]

ground_truth = x["answers"][0] if isinstance(x.get("answers"), list) and len(x["answers"]) else x.get("pred", "")
locality_inputs = {"neighborhood": {"prompt": [x["loc"]], "ground_truth": [x["loc_ans"]]}}
portability_inputs = {"one_hop": {"prompt": [x["portability"]["New Question"]], "ground_truth": [x["portability"]["New Answer"]]}}

metrics, edited_model, _ = editor.edit(
    prompts=[x["src"]],
    rephrase_prompts=[x["rephrase"]],
    subject=[x["subject"]],
    ground_truth=[ground_truth],
    target_new=[x["alt"]],
    locality_inputs=locality_inputs,
    portability_inputs=portability_inputs,
    keep_original_weight=True,
    test_generation=False,
)
print(metrics)
del edited_model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


Torch file in smoke test: /usr/local/lib/python3.12/dist-packages/torch/__init__.py



100%|██████████| 1/1 [00:01<00:00,  1.61s/it]

  0%|          | 0/1 [00:00<?, ?it/s]

Executing ROME algorithm for the update: [When was the inception of IAAF Combined Events Challenge?] -> [ 2006]
Cached context templates ['{}']
Computing left vector (u)...
Selected u projection object IAAF Combined Events Challenge
Left vector shape: torch.Size([9216])
Computing right vector (v)
Lookup index found: 9 | Sentence: When was the inception of IAAF Combined Events Challenge? 200 | Token:  Challenge
Rewrite layer is 17
Tying optimization objective to 31
Recording initial value of v*
loss 3.435 = 3.435 + 0.0 + 0.0 avg prob of [ 2006] 0.032220374792814255
loss 1.702 = 1.667 + 0.034 + 0.001 avg prob of [ 2006] 0.18883256614208221
loss 0.798 = 0.753 + 0.044 + 0.001 avg prob of [ 2006] 0.4709703028202057
loss 0.485 = 0.435 + 0.049 + 0.001 avg prob of [ 2006] 0.6473275423049927
loss 0.374 = 0.31 + 0.063 + 0.001 avg prob of [ 2006] 0.7332613468170166
loss 0.183 = 0.135 + 0.047 + 0.001 avg prob of [ 2006] 0.873528778553009
loss 0.092 = 0.044 + 0.047 + 0.001 avg prob of [ 2006] 0.956

2026-05-05 03:16:45,446 - easyeditor.editors.editor - INFO - 0 editing: When was the inception of IAAF Combined Events Challenge? -> 2006  

 {'pre': {'rewrite_acc': [np.float64(0.2)], 'portability': {'one_hop_acc': [np.float64(0.0)]}, 'rephrase_acc': [np.float64(0.2)]}, 'case_id': 0, 'requested_rewrite': {'prompt': 'When was the inception of IAAF Combined Events Challenge?', 'target_new': '2006', 'ground_truth': '1998', 'portability': {'one_hop': {'prompt': 'What type of sports event is the IAAF Combined Events Challenge, which was established in 2006?', 'ground_truth': 'Athletics'}}, 'locality': {'neighborhood': {'prompt': 'nq question: what is the name of the last episode of spongebob', 'ground_truth': 'The String'}}, 'subject': 'IAAF Combined Events Challenge', 'rephrase_prompt': 'When was the IAAF Combined Events Challenge launched?'}, 'post': {'rewrite_acc': [np.float64(1.0)], 'locality': {'neighborhood_acc': [np.float64(1.0)]}, 'portability': {'one_hop_acc': [np.float64(0.0)]}, 

Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.2), 'rephrase_acc': np.float64(0.2), 'portability': {'one_hop_acc': np.float64(0.0)}}, 'post': {'rewrite_acc': np.float64(1.0), 'rephrase_acc': np.float64(0.6), 'locality': {'neighborhood_acc': np.float64(1.0)}, 'portability': {'one_hop_acc': np.float64(0.0)}}}
[{'pre': {'rewrite_acc': [np.float64(0.2)], 'portability': {'one_hop_acc': [np.float64(0.0)]}, 'rephrase_acc': [np.float64(0.2)]}, 'case_id': 0, 'requested_rewrite': {'prompt': 'When was the inception of IAAF Combined Events Challenge?', 'target_new': '2006', 'ground_truth': '1998', 'portability': {'one_hop': {'prompt': 'What type of sports event is the IAAF Combined Events Challenge, which was established in 2006?', 'ground_truth': 'Athletics'}}, 'locality': {'neighborhood': {'prompt': 'nq question: what is the name of the last episode of spongebob', 'ground_truth': 'The String'}}, 'subject': 'IAAF Combined Events Challenge', 'rephrase_prompt': 'When was the IAAF Combined E

In [14]:
# Cell 10 - N-sample eval loop. Start with N=5, then increase after the smoke test works.
import numpy as np
from pathlib import Path
torch.backends.cudnn.enabled = False

N = 100
out_path = Path(f"/kaggle/working/rome_qwen35_4b_base_zsre_eval_metrics_{N}.jsonl")
json_out_path = Path(f"/kaggle/working/rome_qwen35_4b_base_zsre_eval_metrics_{N}.json")

def scalarize(x):
    if isinstance(x, np.generic):
        return x.item()
    if isinstance(x, list):
        return [scalarize(v) for v in x]
    if isinstance(x, dict):
        return {k: scalarize(v) for k, v in x.items()}
    return x

def first_float(v):
    if isinstance(v, list):
        return float(v[0]) if len(v) else None
    if v is None:
        return None
    return float(v)

def summarize(metrics_list):
    accuracy, generality, locality, portability, runtime = [], [], [], [], []
    for m in metrics_list:
        post = m.get("post", {})
        if "rewrite_acc" in post:
            v = first_float(post["rewrite_acc"])
            if v is not None: accuracy.append(v)
        if "rephrase_acc" in post:
            v = first_float(post["rephrase_acc"])
            if v is not None: generality.append(v)
        for k, v in post.get("locality", {}).items():
            if "acc" in k:
                fv = first_float(v)
                if fv is not None: locality.append(fv)
        for k, v in post.get("portability", {}).items():
            if "acc" in k:
                fv = first_float(v)
                if fv is not None: portability.append(fv)
        if "runtime_seconds_wall" in m:
            runtime.append(float(m["runtime_seconds_wall"]))
    return {
        "n": len(metrics_list),
        "accuracy": sum(accuracy) / len(accuracy) if accuracy else None,
        "generality": sum(generality) / len(generality) if generality else None,
        "locality": sum(locality) / len(locality) if locality else None,
        "portability": sum(portability) / len(portability) if portability else None,
        "avg_runtime_seconds_per_edit": sum(runtime) / len(runtime) if runtime else None,
        "total_runtime_seconds": sum(runtime) if runtime else None,
    }

all_metrics = []
out_path.write_text("")

for i, x in enumerate(eval_items[:N]):
    print(f"\n=== Qwen ROME eval item {i+1}/{N} ===")
    ground_truth = x["answers"][0] if isinstance(x.get("answers"), list) and len(x["answers"]) else x.get("pred", "")
    locality_inputs = {"neighborhood": {"prompt": [x["loc"]], "ground_truth": [x["loc_ans"]]}}
    portability_inputs = {"one_hop": {"prompt": [x["portability"]["New Question"]], "ground_truth": [x["portability"]["New Answer"]]}} if x.get("portability") else {}
    try:
        start = time.perf_counter()
        metrics, edited_model, _ = editor.edit(
            prompts=[x["src"]],
            rephrase_prompts=[x["rephrase"]],
            subject=[x["subject"]],
            ground_truth=[ground_truth],
            target_new=[x["alt"]],
            locality_inputs=locality_inputs,
            portability_inputs=portability_inputs,
            keep_original_weight=True,
            test_generation=False,
        )
        elapsed = time.perf_counter() - start
        m = scalarize(metrics[0] if isinstance(metrics, list) else metrics)
        m["runtime_seconds_wall"] = elapsed
        all_metrics.append(m)
        with open(out_path, "a") as f:
            f.write(json.dumps(m) + "\n")
        print("post:", m.get("post", {}))
        print("running summary:", summarize(all_metrics))
        del edited_model
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    except torch.cuda.OutOfMemoryError:
        print("OOM at item", i)
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        break
    except Exception as e:
        print("Error at item", i, repr(e))
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        continue

summary = summarize(all_metrics)
with open(json_out_path, "w") as f:
    json.dump({"method": "ROME", "model": MODEL_ID, "summary": summary, "metrics": all_metrics}, f, indent=2)
print("Final summary:", summary)
print("Saved:", json_out_path)



=== Qwen ROME eval item 1/100 ===


  0%|          | 0/1 [00:00<?, ?it/s]

Executing ROME algorithm for the update: [When was the inception of IAAF Combined Events Challenge?] -> [ 2006]
Computing left vector (u)...
Selected u projection object IAAF Combined Events Challenge
Left vector shape: torch.Size([9216])
Computing right vector (v)
Lookup index found: 9 | Sentence: When was the inception of IAAF Combined Events Challenge? 200 | Token:  Challenge
Rewrite layer is 17
Tying optimization objective to 31
Recording initial value of v*
loss 3.435 = 3.435 + 0.0 + 0.0 avg prob of [ 2006] 0.032220374792814255
loss 1.702 = 1.667 + 0.034 + 0.001 avg prob of [ 2006] 0.18883256614208221
loss 0.798 = 0.753 + 0.044 + 0.001 avg prob of [ 2006] 0.4709703028202057
loss 0.485 = 0.435 + 0.049 + 0.001 avg prob of [ 2006] 0.6473275423049927
loss 0.374 = 0.31 + 0.063 + 0.001 avg prob of [ 2006] 0.7332613468170166
loss 0.183 = 0.135 + 0.047 + 0.001 avg prob of [ 2006] 0.873528778553009
loss 0.092 = 0.044 + 0.047 + 0.001 avg prob of [ 2006] 0.9568566679954529
loss 0.055 = 0.012

2026-05-05 03:25:16,057 - easyeditor.editors.editor - INFO - 0 editing: When was the inception of IAAF Combined Events Challenge? -> 2006  

 {'pre': {'rewrite_acc': [np.float64(0.2)], 'portability': {'one_hop_acc': [np.float64(0.0)]}, 'rephrase_acc': [np.float64(0.2)]}, 'case_id': 0, 'requested_rewrite': {'prompt': 'When was the inception of IAAF Combined Events Challenge?', 'target_new': '2006', 'ground_truth': '1998', 'portability': {'one_hop': {'prompt': 'What type of sports event is the IAAF Combined Events Challenge, which was established in 2006?', 'ground_truth': 'Athletics'}}, 'locality': {'neighborhood': {'prompt': 'nq question: what is the name of the last episode of spongebob', 'ground_truth': 'The String'}}, 'subject': 'IAAF Combined Events Challenge', 'rephrase_prompt': 'When was the IAAF Combined Events Challenge launched?'}, 'post': {'rewrite_acc': [np.float64(1.0)], 'locality': {'neighborhood_acc': [np.float64(1.0)]}, 'portability': {'one_hop_acc': [np.float64(0.0)]}, 

Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.2), 'rephrase_acc': np.float64(0.2), 'portability': {'one_hop_acc': np.float64(0.0)}}, 'post': {'rewrite_acc': np.float64(1.0), 'rephrase_acc': np.float64(0.6), 'locality': {'neighborhood_acc': np.float64(1.0)}, 'portability': {'one_hop_acc': np.float64(0.0)}}}
post: {'rewrite_acc': [1.0], 'locality': {'neighborhood_acc': [1.0]}, 'portability': {'one_hop_acc': [0.0]}, 'rephrase_acc': [0.6]}
running summary: {'n': 1, 'accuracy': 1.0, 'generality': 0.6, 'locality': 1.0, 'portability': 0.0, 'avg_runtime_seconds_per_edit': 8.056940953999401, 'total_runtime_seconds': 8.056940953999401}

=== Qwen ROME eval item 2/100 ===


...

Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.0), 'rephrase_acc': np.float64(0.0), 'portability': {'one_hop_acc': np.float64(0.5)}}, 'post': {'rewrite_acc': np.float64(1.0), 'rephrase_acc': np.float64(1.0), 'locality': {'neighborhood_acc': np.float64(1.0)}, 'portability': {'one_hop_acc': np.float64(0.5)}}}
post: {'rewrite_acc': [1.0], 'locality': {'neighborhood_acc': [1.0]}, 'portability': {'one_hop_acc': [0.5]}, 'rephrase_acc': [1.0]}
running summary: {'n': 99, 'accuracy': 1.0, 'generality': 0.7898589065255732, 'locality': 0.9803631553631553, 'portability': 0.4892229750925403, 'avg_runtime_seconds_per_edit': 9.181211490424294, 'total_runtime_seconds': 908.9399375520052}

=== Qwen ROME eval item 100/100 ===


  0%|          | 0/1 [00:00<?, ?it/s]

Executing ROME algorithm for the update: [Who is listed as Thomas of Lancaster, 1st Duke of Clarence father?] -> [ Thomas of Lancaster, 1st Duke of Clarence]
Computing left vector (u)...
Selected u projection object Thomas of Lancaster, 1st Duke of Clarence
Left vector shape: torch.Size([9216])
Computing right vector (v)
Lookup index found: 13 | Sentence: Who is listed as Thomas of Lancaster, 1st Duke of Clarence father? Thomas of Lancaster, 1st Duke of | Token:  Clarence
Rewrite layer is 17
Tying optimization objective to 31
Recording initial value of v*
loss 0.192 = 0.192 + 0.0 + 0.0 avg prob of [ Thomas of Lancaster, 1st Duke of Clarence] 0.8255120515823364
loss 0.118 = 0.115 + 0.003 + 0.001 avg prob of [ Thomas of Lancaster, 1st Duke of Clarence] 0.891704261302948
loss 0.128 = 0.08 + 0.047 + 0.001 avg prob of [ Thomas of Lancaster, 1st Duke of Clarence] 0.9231398701667786
loss 0.065 = 0.057 + 0.007 + 0.001 avg prob of [ Thomas of Lancaster, 1st Duke of Clarence] 0.9444307088851929


2026-05-05 03:40:53,658 - easyeditor.editors.editor - INFO - 0 editing: Who is listed as Thomas of Lancaster, 1st Duke of Clarence father? -> Thomas of Lancaster, 1st Duke of Clarence  

 {'pre': {'rewrite_acc': [np.float64(0.9)], 'portability': {'one_hop_acc': [np.float64(0.0)]}, 'rephrase_acc': [np.float64(0.9)]}, 'case_id': 0, 'requested_rewrite': {'prompt': 'Who is listed as Thomas of Lancaster, 1st Duke of Clarence father?', 'target_new': 'Thomas of Lancaster, 1st Duke of Clarence', 'ground_truth': 'Henry IV of England', 'portability': {'one_hop': {'prompt': 'Who was the ruling monarch when Thomas of Lancaster, 1st Duke of Clarence, was alive?', 'ground_truth': 'Henry V'}}, 'locality': {'neighborhood': {'prompt': 'nq question: what happens to water that infiltrates the soil if it is not absorbed by the roots of plants', 'ground_truth': 'runoff'}}, 'subject': 'Thomas of Lancaster, 1st Duke of Clarence', 'rephrase_prompt': "Who is listed as Thomas of Lancaster, 1st Duke of Clarence'

Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.9), 'rephrase_acc': np.float64(0.9), 'portability': {'one_hop_acc': np.float64(0.0)}}, 'post': {'rewrite_acc': np.float64(1.0), 'rephrase_acc': np.float64(1.0), 'locality': {'neighborhood_acc': np.float64(1.0)}, 'portability': {'one_hop_acc': np.float64(0.0)}}}
post: {'rewrite_acc': [1.0], 'locality': {'neighborhood_acc': [1.0]}, 'portability': {'one_hop_acc': [0.0]}, 'rephrase_acc': [1.0]}
running summary: {'n': 100, 'accuracy': 1.0, 'generality': 0.7919603174603175, 'locality': 0.9805595238095237, 'portability': 0.4843307453416149, 'avg_runtime_seconds_per_edit': 9.165970445500053, 'total_runtime_seconds': 916.5970445500052}
Final summary: {'n': 100, 'accuracy': 1.0, 'generality': 0.7919603174603175, 'locality': 0.9805595238095237, 'portability': 0.4843307453416149, 'avg_runtime_seconds_per_edit': 9.165970445500053, 'total_runtime_seconds': 916.5970445500052}
Saved: /kaggle/working/rome_qwen35_4b_base_zsre_eval_metrics_100.json


In [15]:
# Cell 11 - Final compact table
import pandas as pd

data = json.load(open(json_out_path))
s = data["summary"]
df = pd.DataFrame([{
    "method": data.get("method", "ROME"),
    "model": data.get("model", "Qwen/Qwen3.5-4B-Base"),
    "n": s.get("n"),
    "accuracy": s.get("accuracy"),
    "generality": s.get("generality"),
    "locality": s.get("locality"),
    "portability": s.get("portability"),
    "avg_runtime_seconds_per_edit": s.get("avg_runtime_seconds_per_edit"),
}])
display(df)
out_csv = "/kaggle/working/rome_qwen35_4b_base_final_metrics.csv"
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)


,method,model,n,accuracy,generality,locality,portability,avg_runtime_seconds_per_edit
0,ROME,Qwen/Qwen3.5-4B-Base,100,1.0,0.79196,0.98056,0.484331,9.16597


Saved: /kaggle/working/rome_qwen35_4b_base_final_metrics.csv
